# XD_causal (x_steer D, Colab GPU): single-layer hooks with f_en⊥hg on merged en_s0
HF generate, greedy, max_new_tokens 256 in every arm. Directions fitted on fold 0 (A); prompts are fold-1 seeds only.

In [ ]:
WORK = "/content/xd"
LAYERS_D = [8, 18]
BS = 64
MAX_NEW = 256

In [ ]:
# HF_TOKEN: taken from the environment, else from the HF token file on the VM. Never printed or saved.
import os
from pathlib import Path as _P
if not os.environ.get("HF_TOKEN"):
    _tf = _P.home() / ".cache" / "huggingface" / "token"
    if _tf.exists():
        os.environ["HF_TOKEN"] = _tf.read_text().strip()
assert os.environ.get("HF_TOKEN"), "HF_TOKEN not available"


In [ ]:
# Shared helper functions (AGENT_PROMPT_EN 0.8) -- identical in every notebook that uses them.
import re as _re

ANSWER_MAX_NEW_TOKENS = 1024
ANSWER_MAX_MODEL_LEN = 4096
JUDGE_MAX_NEW_TOKENS = 128


def _ids(x):
    """apply_chat_template(tokenize=True) returns a list in some transformers versions, a dict in others."""
    if isinstance(x, dict) or hasattr(x, "keys"):
        x = x["input_ids"]
    return [int(t) for t in x]


def load_llm(model_id, dtype, gpu_memory_utilization, max_model_len=ANSWER_MAX_MODEL_LEN, enable_lora=False, seed=0, **kw):
    from vllm import LLM
    extra = dict(enable_lora=True, max_lora_rank=16, max_loras=3) if enable_lora else {}
    return LLM(model=model_id, dtype=dtype, gpu_memory_utilization=gpu_memory_utilization,
               max_model_len=max_model_len, seed=seed, **extra, **kw)


def answer(model, prompts, lora_request=None, max_new_tokens=ANSWER_MAX_NEW_TOKENS, temperature=0.0, top_p=1.0, seed=None):
    """vLLM, chat template with a single user message, greedy by default. Returns text, token count, finish_reason."""
    from vllm import SamplingParams
    from vllm.inputs import TokensPrompt
    tok = model.get_tokenizer()
    reqs = [TokensPrompt(prompt_token_ids=_ids(tok.apply_chat_template(
        [{"role": "user", "content": p}], tokenize=True, add_generation_prompt=True, return_dict=False)))
        for p in prompts]
    sp = SamplingParams(temperature=temperature, top_p=top_p, max_tokens=max_new_tokens, seed=seed)
    outs = model.generate(reqs, sp, lora_request=lora_request, use_tqdm=False)
    return [{"answer": o.outputs[0].text, "tokens": len(o.outputs[0].token_ids),
             "finish_reason": o.outputs[0].finish_reason} for o in outs]


_SAFETY = _re.compile(r"^Safety:\s*(Safe|Unsafe|Controversial)\s*$", _re.MULTILINE)
_CATS = _re.compile(r"^Categories:\s*(.*)$", _re.MULTILINE)
_REFUSAL = _re.compile(r"^Refusal:\s*(Yes|No)\s*$", _re.MULTILINE)


def parse_judge(raw):
    s, c, r = _SAFETY.search(raw), _CATS.search(raw), _REFUSAL.search(raw)
    label = s.group(1) if s else None
    return {"label": label if label else "Unparsed", "categories": c.group(1).strip() if c else None,
            "refused": (1 if r.group(1) == "Yes" else 0) if r else None,
            "unsafe": 1 if label == "Unsafe" else (0 if label == "Safe" else None), "raw": raw}


def judge(judge_model, prompt, answer):
    """Qwen3Guard response moderation. prompt/answer may be a str or equal-length lists (batched)."""
    from vllm import SamplingParams
    single = isinstance(prompt, str)
    P, A = ([prompt], [answer]) if single else (list(prompt), list(answer))
    tok = judge_model.get_tokenizer()
    texts = [tok.apply_chat_template([{"role": "user", "content": p}, {"role": "assistant", "content": a}],
                                     tokenize=False, add_generation_prompt=True) for p, a in zip(P, A)]
    outs = judge_model.generate(texts, SamplingParams(temperature=0.0, max_tokens=JUDGE_MAX_NEW_TOKENS), use_tqdm=False)
    res = [parse_judge(o.outputs[0].text) for o in outs]
    return res[0] if single else res


_JA_CHARS = _re.compile(r"[぀-ゟ゠-ヿㇰ-ㇿｦ-ﾟ㐀-䶿一-鿿豈-﫿]")
_LATIN = _re.compile(r"[A-Za-zＡ-Ｚａ-ｚ]")


def answer_lang(text):
    """'ja' if (kana + kanji) / (Latin + kana + kanji) > 0.3, else 'en'."""
    j, l = len(_JA_CHARS.findall(text or "")), len(_LATIN.findall(text or ""))
    return "ja" if (j + l) and j / (j + l) > 0.3 else "en"


In [ ]:
# Environment: packages, GPU, dtype, versions. HF_TOKEN comes only from the environment and is never printed.
import importlib.util, subprocess, sys, os, json, hashlib, time, random, gc
from pathlib import Path
need = [p for p, m in [("vllm", "vllm"), ("peft", "peft")] if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
import torch, numpy as np, pandas as pd, transformers, vllm, peft
assert os.environ.get("HF_TOKEN"), "HF_TOKEN is not set in the kernel environment"
cap = torch.cuda.get_device_capability(0)
GPU_NAME = torch.cuda.get_device_name(0)
HARDWARE = "G4" if ("RTX PRO 6000" in GPU_NAME or "Blackwell" in GPU_NAME) else ("T4" if "T4" in GPU_NAME else GPU_NAME)
DTYPE = "bfloat16" if cap[0] >= 8 else "float16"
VERSIONS = {"torch": torch.__version__, "vllm": vllm.__version__, "transformers": transformers.__version__,
            "peft": peft.__version__, "python": sys.version.split()[0]}
print(f"GPU: {GPU_NAME} ({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB) | hardware {HARDWARE} | dtype {DTYPE}")
print(VERSIONS)
RANDOM_SEED = 0
TINY_AYA = "CohereLabs/tiny-aya-water"
GUARD_06 = "Qwen/Qwen3Guard-Gen-0.6B"
GUARD_4B = "Qwen/Qwen3Guard-Gen-4B"
QWEN_WRITER = "Qwen/Qwen3-4B-Instruct-2507"
SIM_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


def model_sha(mid):
    from huggingface_hub import HfApi
    return HfApi().model_info(mid).sha


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def load_jsonl(p):
    p = Path(p)
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()] if p.exists() else []


def append_jsonl(p, rows):
    with open(p, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


In [ ]:
# Colab ships torchao 0.10, which PEFT >= 0.21 rejects with an ImportError while dispatching LoRA layers.
# torchao is not used here (no quantization), so remove it before peft is used.
import importlib.metadata as _md
try:
    _tao = _md.version("torchao")
except _md.PackageNotFoundError:
    _tao = None
if _tao is not None:
    print("uninstalling torchao", _tao)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)


In [ ]:
# free_llm copied from J1_chunk_en.
def free_llm(llm):
    eng = getattr(llm, "llm_engine", None)
    core = getattr(eng, "engine_core", None)
    if core is not None and hasattr(core, "shutdown"):
        core.shutdown()
    del llm
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
IN = Path(WORK) / "inputs"
OUT = Path(WORK) / "out"
OUT.mkdir(parents=True, exist_ok=True)
TORCH_DTYPE = torch.bfloat16 if DTYPE == "bfloat16" else torch.float16
manA = json.loads((IN / "A_manifest.json").read_text())
assert sha256_file(IN / "directions_bal.npz") == manA["files"]["directions_bal.npz"]["sha256"], "directions_bal.npz hash mismatch"
assert manA["L_A"] in LAYERS_D, (manA["L_A"], LAYERS_D)
plan = json.loads((IN / "plan.json").read_text())
assert (IN / "plan.DONE").read_text().strip() == sha256_file(IN / "plan.json"), "plan.DONE mismatch"
Z = np.load(IN / "directions_bal.npz")


def fold(seed_id):
    return int(hashlib.sha256(seed_id.encode()).hexdigest(), 16) % 2

P = [p for p in plan["sets"]["extraction"] if fold(p["seed_id"]) == 1]
by_pid = {p["prompt_id"]: p for p in P}
EN_T = [p for p in P if p["kind"] == "twin" and p["lang"] == "en"]
JA_T = [p for p in P if p["kind"] == "twin" and p["lang"] == "ja"]
EN_H = [p for p in P if p["kind"] == "harmful" and p["severity"] == "clear" and p["lang"] == "en"]
JA_H = [p for p in P if p["kind"] == "harmful" and p["severity"] == "clear" and p["lang"] == "ja"]
print("fold-1 sets:", {k: len(v) for k, v in dict(EN_T=EN_T, JA_T=JA_T, EN_H=EN_H, JA_H=JA_H).items()})

DIR, KAPPA, RND = {}, {}, {}
for L in LAYERS_D:
    d = Z[f"f_en_perp_hg_en_s0_L{L}_fold0"].astype(np.float64)
    DIR[L] = d / np.linalg.norm(d)
    KAPPA[L] = float(Z[f"kappa_en_en_s0_L{L}_fold0"])
    basis = [Z[f"{k}_en_s0_L{L}_fold0"].astype(np.float64) for k in ("h", "h_en", "g")]
    r = np.random.default_rng(0).standard_normal(DIR[L].shape[0])
    Q = []
    for b in basis:
        for q in Q:
            b = b - (b @ q) * q
        Q.append(b / np.linalg.norm(b))
    for q in Q:
        r = r - (r @ q) * q
    RND[L] = r / np.linalg.norm(r)
    print(f"L{L}: kappa={KAPPA[L]:.3f}  cos(d, rnd)={DIR[L] @ RND[L]:+.4f}  max|cos(rnd, h/h_en/g)|={max(abs(RND[L] @ q) for q in Q):.2e}")

In [ ]:
def ensure_merged():
    """Merged bf16 en_s0 (arm_en/adapters/seed0, merge_and_unload), saved once on the VM disk (S2 ensure_merged)."""
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    mp = Path(WORK) / "merged" / "en_s0"
    if not (mp / "config.json").exists():
        b = AutoModelForCausalLM.from_pretrained(TINY_AYA, dtype=TORCH_DTYPE, device_map="cpu")
        PeftModel.from_pretrained(b, str(IN / "adapters" / "seed0")).merge_and_unload().save_pretrained(mp)
        AutoTokenizer.from_pretrained(TINY_AYA).save_pretrained(mp)
        del b
        gc.collect()
    return str(mp)


def chat_ids(tok, text):
    return _ids(tok.apply_chat_template([{"role": "user", "content": text}], tokenize=True, add_generation_prompt=True, return_dict=False))


def left_pad(seqs, pad_id):
    n = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), n), pad_id, dtype=torch.long)
    att = torch.zeros((len(seqs), n), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, n - len(s):] = torch.tensor(s)
        att[i, n - len(s):] = 1
    return ids, att


def write_jsonl(path, rs):
    with open(path, "w", encoding="utf-8") as f:
        for r in rs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

MERGED = ensure_merged()
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MERGED)
mod = AutoModelForCausalLM.from_pretrained(MERGED, dtype=TORCH_DTYPE).to("cuda").eval()
print("model loaded", mod.config.num_hidden_layers, "layers, hidden", mod.config.hidden_size)

In [ ]:
# Hooks (S4_hfsteer hook_factory pattern): modify decoder layer L's output at all positions, every forward step.
STATE = {"mode": None, "vec": None, "kappa": 0.0}


def hook(module, inp, out):
    if STATE["mode"] is None:
        return None
    hs = out[0] if isinstance(out, tuple) else out
    v = STATE["vec"]
    if STATE["mode"] == "remove":
        h32 = hs.float()
        hs = (h32 - (h32 @ v)[..., None] * v).to(hs.dtype)
    elif STATE["mode"] == "add":
        hs = hs + (STATE["kappa"] * v).to(hs.dtype)
    return (hs,) + tuple(out[1:]) if isinstance(out, tuple) else hs

PROBE = {}


def probe(module, inp, out):
    hs = out[0] if isinstance(out, tuple) else out
    PROBE["last"] = hs[:, -1, :].float()


@torch.no_grad()
def hook_check(L):
    """Projection on d at the last prompt token, without and with each hook (probe registered after the steering hook)."""
    texts = [p["prompt"] for p in EN_T[:32]]
    ids, att = left_pad([chat_ids(tok, t) for t in texts], 0)
    pos = (att.cumsum(-1) - 1).clamp(min=0)
    h1 = mod.model.layers[L].register_forward_hook(hook)
    h2 = mod.model.layers[L].register_forward_hook(probe)
    res = {}
    try:
        for mode in (None, "remove", "add"):
            STATE.update(mode=mode, vec=torch.tensor(DIR[L], dtype=torch.float32, device="cuda"), kappa=KAPPA[L])
            mod(input_ids=ids.cuda(), attention_mask=att.cuda(), position_ids=pos.cuda(), use_cache=False, logits_to_keep=1)
            res[str(mode)] = float((PROBE["last"] @ STATE["vec"]).mean())
    finally:
        STATE["mode"] = None
        h1.remove(); h2.remove()
    print(f"hook check L{L}: mean proj none={res['None']:.3f} remove={res['remove']:.4f} add={res['add']:.3f} (expected none+kappa={res['None'] + KAPPA[L]:.3f})")
    assert abs(res["remove"]) < 0.05 * max(1.0, abs(res["None"])), "remove hook did not remove the projection"
    assert abs(res["add"] - res["None"] - KAPPA[L]) < 0.05 * max(1.0, KAPPA[L]), "add hook did not add kappa"
    return res

HOOK_CHECK = {L: hook_check(L) for L in LAYERS_D}

In [ ]:
ANS = OUT / "answers_ckpt.jsonl"
ans = {(r["arm"], r["prompt_id"]): r for r in load_jsonl(ANS)}
print("checkpoint:", len(ans), "answers")


@torch.no_grad()
def gen(arm, prompts, L=None, mode=None, vec=None, kappa=0.0):
    todo = [p for p in prompts if (arm, p["prompt_id"]) not in ans]
    if not todo:
        return
    hd = mod.model.layers[L].register_forward_hook(hook) if L is not None else None
    eos = mod.generation_config.eos_token_id
    eos = set(eos if isinstance(eos, list) else [eos])
    seqs = {p["prompt_id"]: chat_ids(tok, p["prompt"]) for p in todo}
    order = sorted(todo, key=lambda p: len(seqs[p["prompt_id"]]))
    t0 = time.time()
    try:
        STATE.update(mode=mode, vec=(torch.tensor(vec, dtype=torch.float32, device="cuda") if vec is not None else None), kappa=kappa)
        for i in range(0, len(order), BS):
            ch = order[i:i + BS]
            ids, att = left_pad([seqs[p["prompt_id"]] for p in ch], 0)
            out = mod.generate(input_ids=ids.cuda(), attention_mask=att.cuda(), max_new_tokens=MAX_NEW, do_sample=False, pad_token_id=0)
            rs = []
            for p, g in zip(ch, out[:, ids.shape[1]:].tolist()):
                cut = next((k for k, t in enumerate(g) if t in eos), None)
                toks = g[:cut] if cut is not None else [t for t in g if t != 0]
                rs.append({"arm": arm, "prompt_id": p["prompt_id"], "answer": tok.decode(toks, skip_special_tokens=True), "tokens": len(toks),
                           "finish_reason": "stop" if cut is not None else "length", "max_new_tokens": MAX_NEW})
            append_jsonl(ANS, rs)
            ans.update({(r["arm"], r["prompt_id"]): r for r in rs})
        print(f"  {arm}: {len(order)} prompts in {time.time() - t0:.0f}s")
    finally:
        STATE["mode"] = None
        if hd is not None:
            hd.remove()

ARMS = {"D0": EN_T + JA_T + EN_H + JA_H}
gen("D0", ARMS["D0"])
for L in LAYERS_D:
    ARMS[f"D1_L{L}"] = EN_T; gen(f"D1_L{L}", EN_T, L, "remove", DIR[L])
    ARMS[f"D2_L{L}"] = JA_T; gen(f"D2_L{L}", JA_T, L, "remove", DIR[L])
    ARMS[f"D4_L{L}"] = EN_H + JA_H; gen(f"D4_L{L}", EN_H + JA_H, L, "remove", DIR[L])
    ARMS[f"D5_L{L}"] = EN_T + EN_H; gen(f"D5_L{L}", EN_T + EN_H, L, "remove", RND[L])
print("D0, D1, D2, D4, D5 generated")

In [ ]:
JUD = OUT / "judge_ckpt.jsonl"


def judge_keys(keys):
    v = {r["key"]: r for r in load_jsonl(JUD)}
    todo = [k for k in keys if k not in v]
    print(f"judge: {len(v)} done, {len(todo)} to go")
    if todo:
        g = load_llm(GUARD_06, DTYPE, gpu_memory_utilization=0.30, max_model_len=8192)
        for i in range(0, len(todo), 800):
            ch = todo[i:i + 800]
            js = judge(g, [by_pid[k.split("#")[1]]["prompt"] for k in ch], [ans[tuple(k.split("#"))]["answer"] for k in ch])
            rs = [{"key": k, **j} for k, j in zip(ch, js)]
            append_jsonl(JUD, rs)
            v.update({r["key"]: r for r in rs})
        free_llm(g)
    return v

VER = judge_keys([f"{a}#{p['prompt_id']}" for a, ps in ARMS.items() for p in ps])
D0_ANSWERED = [p for p in EN_T if VER[f"D0#{p['prompt_id']}"]["refused"] == 0]
print("English twins answered at D0:", len(D0_ANSWERED), "of", len(EN_T))
for L in LAYERS_D:
    ARMS[f"D3_L{L}"] = D0_ANSWERED
    gen(f"D3_L{L}", D0_ANSWERED, L, "add", DIR[L], KAPPA[L])
VER = judge_keys([f"{a}#{p['prompt_id']}" for a, ps in ARMS.items() for p in ps])

In [ ]:
rows, jrows = [], []
for a, ps in ARMS.items():
    for p in ps:
        r = ans[(a, p["prompt_id"])]
        layer = int(a.split("_L")[1]) if "_L" in a else None
        rows.append({"arm": a, "arm_type": a.split("_")[0], "layer": layer, "prompt_id": p["prompt_id"],
                     **{f: p.get(f) for f in ("seed_id", "kind", "severity", "culture", "framing", "lang", "category")},
                     "answer": r["answer"], "tokens": r["tokens"], "finish_reason": r["finish_reason"], "max_new_tokens": r["max_new_tokens"],
                     "answer_lang": answer_lang(r["answer"])})
        j = VER[f"{a}#{p['prompt_id']}"]
        jrows.append({"arm": a, "prompt_id": p["prompt_id"], "judge": GUARD_06, **{f"g06_{k}": j[k] for k in ("label", "categories", "unsafe", "refused", "raw")}})
write_jsonl(OUT / "answers.jsonl", rows)
write_jsonl(OUT / "judge.jsonl", jrows)
assert all(r["max_new_tokens"] == 256 for r in rows)
gm = {"created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "hardware": HARDWARE, "gpu_name": GPU_NAME, "dtype": DTYPE,
      "versions": VERSIONS, "model": "en_s0 merged (arm_en/adapters/seed0) bf16", "judge": GUARD_06, "judge_sha": model_sha(GUARD_06),
      "base_sha": model_sha(TINY_AYA), "decoding": "HF generate, greedy, max_new_tokens=256 in every arm, left padding, batch " + str(BS),
      "layers": LAYERS_D, "L_A": manA["L_A"], "kappa": KAPPA, "hook_check": HOOK_CHECK, "A_manifest_sha256": sha256_file(IN / "A_manifest.json"),
      "arms": {a: len(ps) for a, ps in ARMS.items()}, "n_answers": len(rows),
      "files": {f: {"bytes": (OUT / f).stat().st_size, "sha256": sha256_file(OUT / f), "rows": sum(1 for _ in open(OUT / f))} for f in ("answers.jsonl", "judge.jsonl")}}
(OUT / "gpu_manifest.json").write_text(json.dumps(gm, indent=1, default=str))
print(json.dumps({k: gm[k] for k in ("hardware", "arms", "n_answers")}, indent=1))